# LEACE Concept Erasure Recipe: scrub grammatical number out of GPT-2 small

LEAst-squares Concept Erasure (Belrose et al. 2023, "LEACE: Perfect linear concept erasure in closed form", NeurIPS 2023, arXiv:2306.03819) is the smallest affine change to a representation after which no linear classifier can recover a concept. It is closed form: whiten, project out the span of the concept's cross-covariance, unwhiten. The paper's headline use is concept scrubbing: fit an eraser for every layer in turn, each on activations the earlier erasers already changed, and apply them all inside the forward pass.

This notebook scrubs subject number (singular versus plural) out of GPT-2 small with one `tl.when` rule per block, and measures two things: a linear probe for number falls to chance, and the model's subject-verb agreement falls with it, while erasing one block alone lets number re-emerge and a random-direction control of the same rank changes nothing. It downloads GPT-2 small (about 550 MB) at a pinned revision and runs on a laptop CPU in a few minutes.

In [ ]:
# recipe-test: tool=transformers expected=True
from __future__ import annotations

import functools
import operator

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

import torchlens as tl

MODEL_ID = "openai-community/gpt2"
REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # pragma: allowlist secret
torch.manual_seed(0)
model = GPT2LMHeadModel.from_pretrained(
    MODEL_ID, revision=REVISION, attn_implementation="eager", use_cache=False
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
BLOCKS = [f"transformer.h.{layer}" for layer in range(model.config.n_layer)]
IS, ARE = tokenizer.encode(" is")[0], tokenizer.encode(" are")[0]
RESULT = True

## Sentences whose next word must agree with the subject

Thirty-two nouns in eight five-token frames, singular and plural, each ending where the verb goes. Every noun is one GPT-2 token and every frame has the same shape, so position `k` means the same thing in every sentence and the whole set batches without padding.

In [ ]:
# recipe-test: tool=transformers expected=True
NOUN_PAIRS = [
    ("cat", "cats"),
    ("dog", "dogs"),
    ("car", "cars"),
    ("book", "books"),
    ("teacher", "teachers"),
    ("bird", "birds"),
    ("house", "houses"),
    ("tree", "trees"),
    ("key", "keys"),
    ("child", "children"),
    ("boy", "boys"),
    ("girl", "girls"),
    ("door", "doors"),
    ("river", "rivers"),
    ("apple", "apples"),
    ("horse", "horses"),
    ("man", "men"),
    ("woman", "women"),
    ("friend", "friends"),
    ("student", "students"),
    ("flower", "flowers"),
    ("ship", "ships"),
    ("plane", "planes"),
    ("truck", "trucks"),
    ("cup", "cups"),
    ("box", "boxes"),
    ("hat", "hats"),
    ("shoe", "shoes"),
    ("coin", "coins"),
    ("phone", "phones"),
    ("chair", "chairs"),
    ("bottle", "bottles"),
]
FRAMES = [
    "The {} near the window",
    "The {} in the garden",
    "The {} by the river",
    "The {} on the table",
    "The {} behind the house",
    "The {} under the tree",
    "The {} at the station",
    "The {} from the city",
]
nouns = [
    pair for pair in NOUN_PAIRS if all(len(tokenizer.encode(" " + form)) == 1 for form in pair)
]
texts = [frame.format(form) for frame in FRAMES for pair in nouns for form in pair]
ids = torch.tensor([tokenizer.encode(text) for text in texts])  # same length by construction
plural = torch.tensor(
    [float(form == 1) for _frame in FRAMES for _pair in nouns for form in range(2)]
)
print(
    ids.shape[0], "sentences of", ids.shape[1], "tokens;", len(nouns), "noun pairs; e.g.", texts[:2]
)


def run(intervene=None, site: str | None = None) -> tuple[torch.Tensor | None, torch.Tensor]:
    """Trace the set one frame at a time, saving only the head and, if given, ``site``.

    Returns ``site``'s output ``[sentence, position, d]`` (None without a site) and the is/are
    logits ``[sentence, 2]``. ``on_zero_fire="error"`` refuses a rule that matched no site.
    """
    hidden, verbs = [], []
    for batch in ids.split(2 * len(nouns)):  # one frame per batch
        trace = tl.trace(
            model,
            batch,
            intervene=intervene,
            save=["lm_head"] + ([site] if site else []),
            on_zero_fire="error",
        )
        if site:
            hidden.append(trace[site].out.detach())
        verbs.append(trace["lm_head"].out[:, -1, [IS, ARE]].detach())
    return (torch.cat(hidden) if site else None), torch.cat(verbs)


def agreement(verb_logits: torch.Tensor) -> float:
    """Fraction of sentences whose preferred verb form agrees with the subject."""
    return ((verb_logits[:, 1] > verb_logits[:, 0]).float() == plural).float().mean().item()


_, clean_logits = run()
print(f"agreement accuracy, clean model: {agreement(clean_logits):.2f}")
RESULT = ids.ndim == 2

## LEACE in closed form, one eraser per position

With `x` a block's output at one position and `W` the ZCA whitening of `x`, LEACE's projection is `P = W^+ proj(W Sigma_xz) W` and the erased value is `x - P (x - mean)` (Belrose et al. 2023, Theorem 4.1). Fitting one eraser per position keeps the guarantee exact at every position, including the last one, where the verb is decided.

In [ ]:
# recipe-test: tool=transformers expected=True
def fit_leace(x: torch.Tensor, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return ``(mean, P)`` such that ``x - P (x - mean)`` erases ``z`` linearly."""
    x, z = x.double(), z.double()[:, None]
    mean = x.mean(0)
    x_centered, z_centered = x - mean, z - z.mean(0)
    sigma_xx = x_centered.T @ x_centered / len(x)
    sigma_xz = x_centered.T @ z_centered / len(x)
    evals, evecs = torch.linalg.eigh(sigma_xx)
    keep = evals > evals.max() * 1e-8
    vecs, vals = evecs[:, keep], evals[keep]
    whiten = (vecs / vals.sqrt()) @ vecs.T  # Sigma_xx^(-1/2) on the data's span
    unwhiten = (vecs * vals.sqrt()) @ vecs.T  # Sigma_xx^(1/2)
    basis, _ = torch.linalg.qr(whiten @ sigma_xz)  # orthonormal span of W Sigma_xz
    return mean.float(), (unwhiten @ basis @ basis.T @ whiten).float()


def erase(activation: torch.Tensor, maps: list[tuple[torch.Tensor, torch.Tensor]]) -> torch.Tensor:
    """Apply one affine erasure per position to a ``[batch, position, d]`` activation."""
    means = torch.stack([mean for mean, _projection in maps])  # [position, d]
    projections = torch.stack([projection for _mean, projection in maps])  # [position, d, d]
    return activation - torch.einsum("bpd,ped->bpe", activation - means, projections)


def eraser(maps: list[tuple[torch.Tensor, torch.Tensor]]):
    """A ``tl.when`` hook that erases in place of the block's output."""

    def hook(activation: torch.Tensor, *, ctx) -> torch.Tensor:
        return erase(activation, maps)

    return hook


def probe_accuracy(features: torch.Tensor) -> float:
    """Accuracy of a logistic probe for number trained on these very activations."""
    feats = (features - features.mean(0)) / (features.std(0) + 1e-6)
    weight, bias = (
        torch.zeros(feats.shape[1], requires_grad=True),
        torch.zeros(1, requires_grad=True),
    )
    optimizer = torch.optim.LBFGS([weight, bias], max_iter=200, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        logits = feats @ weight + bias
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, plural)
        loss = loss + 1e-2 * weight.square().sum()
        loss.backward()
        return loss

    optimizer.step(closure)
    return (((feats @ weight + bias) > 0).float() == plural).float().mean().item()


RESULT = True

## Scrub every block

Block by block: trace with the erasers fitted so far, fit this block's erasers on what it now outputs, add its rule. Rules compose with `&`, so the finished scrubber is one `intervene=` argument. The probe check reads the eleven-block scrub at the last block and applies that block's eraser here, the same arithmetic the twelfth rule runs inside the forward pass.

In [ ]:
# recipe-test: tool=transformers expected=True
rules = []
erasers = {}
for site in BLOCKS:
    hidden, _ = run(intervene=functools.reduce(operator.and_, rules) if rules else None, site=site)
    erasers[site] = [fit_leace(hidden[:, position], plural) for position in range(ids.shape[1])]
    rules.append(tl.when(tl.module(site), eraser(erasers[site])))
scrub = functools.reduce(operator.and_, rules)

last_block_clean, _ = run(site=BLOCKS[-1])
through_eleven, _ = run(intervene=functools.reduce(operator.and_, rules[:-1]), site=BLOCKS[-1])
last_block_scrubbed = erase(through_eleven, erasers[BLOCKS[-1]])
before = probe_accuracy(last_block_clean[:, -1])
after = probe_accuracy(last_block_scrubbed[:, -1])
print(f"number probe at the last block, last position: clean {before:.2f}; scrubbed {after:.2f}")
assert before > 0.95, "number is linearly readable in the clean model"
assert after < 0.6, (
    "after scrubbing, no linear probe beats a constant, even one trained on these activations"
)
RESULT = before > 0.95 and after < 0.6

## What the model does without the concept

LEACE's guarantee (Theorem 4.1) is about the distribution the eraser was fitted on: no linear predictor of the erased concept beats a constant, so the probe trained on the scrubbed activations themselves lands at chance. That is the check above.

Three forward passes under three rule sets: the full scrub, block 6's erasers alone, and a control that erases one random direction per position in every block, mean-matched, the same rank as the scrub.

In [ ]:
# recipe-test: tool=transformers expected=True
def random_maps(seed: int, reference: list[tuple[torch.Tensor, torch.Tensor]]):
    """Mean-matched rank-1 erasers along random directions, one per position."""
    generator = torch.Generator().manual_seed(seed)
    maps = []
    for mean, _projection in reference:
        direction = torch.nn.functional.normalize(
            torch.randn(mean.shape[0], generator=generator), dim=0
        )
        maps.append((mean, torch.outer(direction, direction)))
    return maps


one_block = tl.when(tl.module(BLOCKS[6]), eraser(erasers[BLOCKS[6]]))
control = functools.reduce(
    operator.and_,
    [
        tl.when(tl.module(site), eraser(random_maps(index, erasers[site])))
        for index, site in enumerate(BLOCKS)
    ],
)
_, scrubbed_logits = run(intervene=scrub)
_, one_block_logits = run(intervene=one_block)
_, control_logits = run(intervene=control)
for name, logits in [
    ("clean", clean_logits),
    ("block 6 only", one_block_logits),
    ("random control", control_logits),
    ("full scrub", scrubbed_logits),
]:
    print(f"agreement accuracy, {name:>15}: {agreement(logits):.2f}")
assert agreement(scrubbed_logits) < agreement(clean_logits) - 0.2, (
    "scrubbing number should break agreement"
)
assert agreement(control_logits) > agreement(clean_logits) - 0.1, (
    "a random direction per block should not"
)
RESULT = (
    agreement(scrubbed_logits) < agreement(clean_logits) - 0.2
    and agreement(control_logits) > agreement(clean_logits) - 0.1
)